# Day 4-5: Fine-Tune Legal-BERT on CUAD

This notebook fine-tunes a Legal-BERT transformer model to classify which
legal clause categories appear in a contract paragraph (multi-label classification).

**Run cells top to bottom, in order. Do not skip any cell.**

By the end you will have:
- A fine-tuned model saved to your Google Drive
- Evaluation metrics (F1, precision, recall) on the held-out test set

## Cell 1: Check GPU
Confirm Colab gave us a GPU. You should see something like `Tesla T4`.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2: Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

## Cell 3: Mount Google Drive
A popup will ask you to sign in and allow access. Click through it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Path to the folder you created in Google Drive
DATA_DIR = '/content/drive/MyDrive/contract-intelligence'
OUTPUT_DIR = '/content/drive/MyDrive/contract-intelligence/model_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Confirm the data files are there
for fname in ['train.jsonl', 'validation.jsonl', 'test.jsonl', 'categories.json']:
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f'  {fname}: {"OK" if exists else "MISSING"} ({size:,} bytes)')

print('\nIf any file says MISSING, check the folder name in Google Drive matches: contract-intelligence')

Mounted at /content/drive
  train.jsonl: OK (51,003,332 bytes)
  validation.jsonl: OK (5,932,212 bytes)
  test.jsonl: OK (6,061,214 bytes)
  categories.json: OK (1,068 bytes)

If any file says MISSING, check the folder name in Google Drive matches: contract-intelligence


## Cell 4: Load the dataset

In [ ]:
import json
from datasets import load_dataset

# Load categories
with open(os.path.join(DATA_DIR, 'categories.json')) as f:
    categories = json.load(f)
num_labels = len(categories)
print(f'Number of clause categories: {num_labels}')
print(f'First 5: {categories[:5]}')

# Load the JSONL splits
data_files = {
    'train':      os.path.join(DATA_DIR, 'train.jsonl'),
    'validation': os.path.join(DATA_DIR, 'validation.jsonl'),
    'test':       os.path.join(DATA_DIR, 'test.jsonl'),
}
raw_ds = load_dataset('json', data_files=data_files)
print('\nDataset loaded:')
print(raw_ds)

Number of clause categories: 41
First 5: ['Affiliate License-Licensee', 'Affiliate License-Licensor', 'Agreement Date', 'Anti-Assignment', 'Audit Rights']


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


Dataset loaded:
DatasetDict({
    train: Dataset({
        features: ['contract_id', 'text', 'labels'],
        num_rows: 32242
    })
    validation: Dataset({
        features: ['contract_id', 'text', 'labels'],
        num_rows: 3756
    })
    test: Dataset({
        features: ['contract_id', 'text', 'labels'],
        num_rows: 3841
    })
})


## Cell 5: Tokenize the text

Converts raw text into numbers the model can read.
`nlpaueb/legal-bert-base-uncased` is a BERT model pre-trained specifically
on legal documents — it understands legal language better than general BERT.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256  # max tokens per chunk (256 is safe for 15GB T4 VRAM)

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
    enc['labels'] = [list(map(float, l)) for l in batch['labels']]
    return enc

print('Tokenizing dataset (this takes a few minutes)...')
tokenized_ds = raw_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=['text', 'contract_id']
)
print('Done.')
print(tokenized_ds)

Loading tokenizer: nlpaueb/legal-bert-base-uncased ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

...

Tokenizing dataset (this takes a few minutes)...


Map:   0%|          | 0/32242 [00:00<?, ? examples/s]

Map:   0%|          | 0/3756 [00:00<?, ? examples/s]

Map:   0%|          | 0/3841 [00:00<?, ? examples/s]

Done.
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 32242
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3756
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3841
    })
})


## Cell 6: Load the model

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f'Loading model: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type='multi_label_classification',
)
print('Model loaded.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

Loading model: nlpaueb/legal-bert-base-uncased ...


[transformers] You passed `num_labels=41` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Model loaded.
Total parameters: 109,513,769


## Cell 7: Define evaluation metrics

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

THRESHOLD = 0.5  # probability above this = positive label

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)
    return {
        'f1_micro':        f1_score(labels, preds, average='micro',  zero_division=0),
        'f1_macro':        f1_score(labels, preds, average='macro',  zero_division=0),
        'precision_micro': precision_score(labels, preds, average='micro', zero_division=0),
        'recall_micro':    recall_score(labels, preds, average='micro',    zero_division=0),
    }

print('Metrics function ready.')

Metrics function ready.


## Cell 8: Configure and run training

This is the main training step. On a T4 GPU expect roughly **40-60 minutes**
for 3 epochs. Do not close the browser tab while it runs.

You will see a progress bar and loss/F1 scores printed after each epoch.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

try:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1_micro',
        logging_steps=100,
        fp16=True,
        report_to='none',
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1_micro',
        logging_steps=100,
        fp16=True,
        report_to='none',
    )

# Fix label dtype: multi-label classification needs Float not Long
import torch
def collate_fn(batch):
    result = data_collator(batch)
    result['labels'] = result['labels'].float()
    return result

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    processing_class=tokenizer,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

print('Starting training...')
print('Expected time: 40-60 minutes on T4 GPU')
print('Do not close this tab.\n')
trainer.train()

Starting training...
Expected time: 40-60 minutes on T4 GPU
Do not close this tab.



Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Precision Micro,Recall Micro
1,0.021402,0.027036,0.673088,0.473843,0.747170,0.612371
2,0.018318,0.025850,0.676436,0.480028,0.783967,0.594845
3,0.015727,0.026134,0.683425,0.504215,0.736310,0.637629


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6048, training_loss=0.019420928316851143, metrics={'train_runtime': 1390.5382, 'train_samples_per_second': 69.56, 'train_steps_per_second': 4.349, 'total_flos': 1.2729286514078388e+16, 'train_loss': 0.019420928316851143, 'epoch': 3.0})

## Cell 9: Evaluate on the held-out test set

This is the honest final score — the model has never seen these contracts.

In [ ]:
print('Evaluating on test set...')
test_metrics = trainer.evaluate(tokenized_ds['test'])

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<30}: {v:.4f}')

# Save metrics to Drive
metrics_path = os.path.join(OUTPUT_DIR, 'test_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'\nMetrics saved to: {metrics_path}')

Evaluating on test set...


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,Precision Micro,Recall Micro
0.015727,0.033825,3,0.670394,0.524172,0.787199,0.583773



TEST SET RESULTS
  eval_loss                     : 0.0338
  eval_f1_micro                 : 0.6704
  eval_f1_macro                 : 0.5242
  eval_precision_micro          : 0.7872
  eval_recall_micro             : 0.5838

Metrics saved to: /content/drive/MyDrive/contract-intelligence/model_output/test_metrics.json


## Cell 10: Save the final model to Google Drive

In [ ]:
final_model_dir = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

# Also save the categories list alongside the model
# so we always know what label index 0, 1, 2... means
import shutil
shutil.copy(
    os.path.join(DATA_DIR, 'categories.json'),
    os.path.join(final_model_dir, 'categories.json')
)

print('Model saved to Google Drive at:')
print(f'  {final_model_dir}')
print('\nFiles saved:')
for f in sorted(os.listdir(final_model_dir)):
    size = os.path.getsize(os.path.join(final_model_dir, f))
    print(f'  {f:<40} {size:>10,} bytes')

print('\nDay 4-5 complete: fine-tuned clause classification model saved.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to Google Drive at:
  /content/drive/MyDrive/contract-intelligence/model_output/final_model

Files saved:
  categories.json                               1,068 bytes
  config.json                                   2,604 bytes
  model.safetensors                        438,078,612 bytes
  tokenizer.json                              701,778 bytes
  tokenizer_config.json                           351 bytes
  training_args.bin                             5,201 bytes

Day 4-5 complete: fine-tuned clause classification model saved.
